# Module 05 — MCP Server

**Goal:** Expose your database tools via the **Model Context Protocol** (MCP) so *any* MCP-compatible client (Claude Desktop, Cursor, your own agents) can use them — without writing agent code for each client.

## The big idea

In Modules 03 and 04 you wrote a custom agent: tool definitions, an executor, a message loop. That code is tightly coupled to the OpenAI SDK and only runs in your Python script.

MCP separates **tool implementation** from **tool consumption**:

```
Without MCP:                        With MCP:
  your_agent.py                       Claude Desktop \
  ├── tool definitions                Cursor IDE     ├─► mcp_server  ─► database
  ├── tool executor                   your_agent.py /
  └── message loop
```

Write the server once → reuse it across every MCP client.

Runs on the **Databricks free tier** against the built-in `samples` catalog. We exercise the server **inside this notebook** using the in-process MCP entry points — no laptop required.

---
**Run each cell in order (Shift+Enter).**

## Step 1 — Install the MCP SDK

In [ ]:
%pip install --quiet "mcp[cli]"
dbutils.library.restartPython()

## Step 2 — Pick the catalog and schema

MCP itself doesn't need an LLM API key — clients (Claude Desktop, Cursor, etc.) bring their own. Just pick which schema the server should expose.

In [ ]:
dbutils.widgets.text("CATALOG", "samples", "Catalog")
dbutils.widgets.text("SCHEMA", "bakehouse", "Schema")

CATALOG = dbutils.widgets.get("CATALOG")
SCHEMA = dbutils.widgets.get("SCHEMA")

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"Database: {CATALOG}.{SCHEMA}")

## Step 3 — Database and safety helpers

These power the tools. The agent or MCP client sees only the tool surface; these helpers stay private.

In [ ]:
import re, json
from dataclasses import dataclass

_FORBIDDEN = {"DROP","DELETE","UPDATE","INSERT","ALTER","TRUNCATE","CREATE","REPLACE","MERGE","EXEC","EXECUTE","GRANT","REVOKE","ATTACH","DETACH"}

@dataclass
class ValidationResult:
    is_safe: bool
    reason: str

def validate_sql(sql: str) -> ValidationResult:
    stripped = sql.strip()
    if not stripped:
        return ValidationResult(False, "SQL is empty.")
    cleaned = stripped.rstrip(";")
    if ";" in cleaned:
        return ValidationResult(False, "Only a single SELECT statement is allowed.")
    if cleaned.split()[0].upper() != "SELECT":
        return ValidationResult(False, f"Query must start with SELECT, got '{cleaned.split()[0]}'.")
    for kw in _FORBIDDEN:
        if re.search(rf"\b{kw}\b", cleaned.upper()):
            return ValidationResult(False, f"Forbidden keyword: {kw}")
    return ValidationResult(True, "ok")

def _get_schema() -> dict:
    schema = {}
    for row in spark.sql("SHOW TABLES").collect():
        table_name = row.tableName
        cols = spark.sql(f"DESCRIBE TABLE {table_name}").collect()
        schema[table_name] = [
            {"name": c.col_name, "type": c.data_type}
            for c in cols
            if c.col_name and not c.col_name.startswith("#")
        ]
    return schema

def _run_query(sql: str):
    df = spark.sql(sql).limit(100)
    columns = df.columns
    rows = [row.asDict() for row in df.collect()]
    return columns, rows

print(f"Tables in {CATALOG}.{SCHEMA}: {list(_get_schema().keys())}")

## Step 4 — Define the MCP server

`FastMCP` is a high-level wrapper around the protocol. You decorate Python functions with `@mcp.tool()` and everything else (JSON-schema generation, request/response envelopes, error handling) is automatic.

Notice the docstrings — they become the tool descriptions that MCP clients see, so they need to be as clear as a prompt.

In [ ]:
from mcp.server.fastmcp import FastMCP

mcp = FastMCP(
    "db-agent",
    instructions=(
        f"You have access to a Spark SQL database at {CATALOG}.{SCHEMA}. "
        "Use list_tables to explore the schema, describe_table for column details, "
        "and run_query to answer questions with data. "
        "Only SELECT queries are allowed."
    ),
)

@mcp.tool()
def list_tables() -> str:
    """List all tables in the database.

    Call this first to understand what data is available before writing any SQL.
    Returns a JSON array of table names.
    """
    return json.dumps(list(_get_schema().keys()), indent=2)


@mcp.tool()
def describe_table(table_name: str) -> str:
    """Get the column names and data types for a specific table.

    Use this after list_tables to understand a table's structure before querying it.
    Returns a JSON array of {name, type} objects.

    Args:
        table_name: Name of the table to describe (must be a valid table name).
    """
    schema = _get_schema()
    if table_name not in schema:
        return json.dumps({
            "error": f"Table '{table_name}' not found.",
            "available_tables": list(schema.keys()),
        })
    return json.dumps(schema[table_name], indent=2)


@mcp.tool()
def run_query(sql: str) -> str:
    """Execute a read-only SELECT query against the database and return the results.

    The query is validated before execution:
      - Must be a single SELECT statement
      - Must not contain any write or admin keywords (DROP, DELETE, INSERT, etc.)

    If the query is blocked or fails, an error message is returned explaining why.
    Results are capped at 100 rows to keep responses manageable.

    Args:
        sql: A valid Spark SQL SELECT statement. Use only tables and columns from the schema.
    """
    validation = validate_sql(sql)
    if not validation.is_safe:
        return json.dumps({"error": "Query blocked by safety check.", "reason": validation.reason})

    try:
        columns, rows = _run_query(sql)
        return json.dumps({"columns": columns, "row_count": len(rows), "rows": rows}, default=str)
    except Exception as exc:
        return json.dumps({"error": "Query execution failed.", "detail": str(exc)})


import asyncio
print("Server defined. Tools registered:")
for tool in asyncio.run(mcp.list_tools()):
    print(f"  - {tool.name}: {tool.description.splitlines()[0]}")

## Step 5 — Exercise the tools through MCP

`mcp.call_tool(name, args)` is the same entry point any MCP client uses to invoke a tool. Under the hood it validates arguments against the JSON schema, calls the function, and wraps the result in an MCP response envelope.

This is exactly what Claude Desktop does when you click the hammer icon.

In [ ]:
async def call(name, **kwargs):
    content, structured = await mcp.call_tool(name, kwargs)
    text = content[0].text if content else ""
    print(f"=== {name}({kwargs}) ===")
    print(text[:600] + ("..." if len(text) > 600 else ""))
    print()

asyncio.run(call("list_tables"))

In [ ]:
_first_table = next(iter(_get_schema().keys()))
asyncio.run(call("describe_table", table_name=_first_table))

In [ ]:
asyncio.run(call("run_query", sql=f"SELECT * FROM {_first_table} LIMIT 3"))

In [ ]:
# Safety check should block this:
asyncio.run(call("run_query", sql=f"DELETE FROM {_first_table}"))

## Step 6 — Deploying the server (outside this notebook)

Inside the notebook we ran the server in-process. Real MCP clients talk to the server over a transport — either **stdio** (Claude Desktop launches a subprocess) or **streamable-http** (a long-running HTTP service).

### Option A — Streamable HTTP (recommended on Databricks)

Deploy the same server code as a **Databricks App** (or any HTTP host) and start it with:

```python
mcp.run(transport="streamable-http", host="0.0.0.0", port=8000)
```

Clients point at `https://<your-app>/mcp`.

### Option B — stdio (for Claude Desktop on a laptop)

Save the tool definitions to `server.py` and add to `~/Library/Application Support/Claude/claude_desktop_config.json`:

```json
{
  "mcpServers": {
    "db-agent": {
      "command": "python",
      "args": ["/path/to/server.py"]
    }
  }
}
```

The teaching point is the same either way: **you wrote the tools once, any MCP client can use them.**

## Summary

| Piece | Role |
|-------|------|
| `@mcp.tool()` decorator | Registers a function as a tool, auto-generates its JSON schema |
| Docstring | Becomes the tool description an MCP client (or LLM) sees |
| `mcp.list_tools()` | Returns the tools an MCP client can discover |
| `mcp.call_tool(name, args)` | Invokes a tool — this is the protocol's core entry point |
| `mcp.run(transport=...)` | Exposes the server over stdio or HTTP for external clients |

Your MCP server = the **tool** (stateless, reusable, protocol-compliant).  
Claude Desktop / Cursor / your agent = the **orchestrator** (decides when and how to use it).

That's the full series. You've built the same capability **five different ways** — now you know which layer to reach for depending on how much control you want to keep vs how much you want the LLM (or the protocol) to handle for you.